# stock_return_facts

> 对应代码：`EconometricsCode/code_in_notes/Chap.28/stock_return_facts.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.28`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.28")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

读入全部股票的价格数据。

In [ ]:
%%stata
clear all
set more off
use ../datasets/stock_price

按股票和日期排序后，在每组股票内部生成行号 t。由于停牌等原因日历日期不规则，用行号作时间索引可避免 L. 算子在非交易日产生缺失值。

In [ ]:
%%stata
// 为了避免非交易日之后的缺失值，用每只股票的行号作为时间
sort stkcd date
gen t=.
by stkcd: replace t=_n
tsset stkcd t

计算对数收益率及其平方。收益率本身近似不相关，但平方序列的自相关是检验 ARCH 效应（波动聚集）的核心工具。

In [ ]:
%%stata
gen rr=log(clsprc)-log(L.clsprc)
label variable rr "对数收益率"
gen rr2=rr^2
label variable rr2 "对数收益率平方"

用 egen 的 count 函数统计每只股票的观测天数，剔除上市不足半年（180 个交易日）的股票，保证后续统计量可靠。

In [ ]:
%%stata
// 去除上市不到半年的股票
egen days=count(stkcd), by(stkcd)
drop if days<180

对三只代表性股票（贵州茅台 600519、浦发银行 600000 中的 66、上海机场 600009）循环绘制五种诊断图：收益率时序图看波动聚集；rr 的 ACF 图验证收益率弱白噪声性质；rr² 的 ACF 图验证 ARCH 效应；Q-Q 图和叠加正态密度的直方图则显示收益率相对正态分布的厚尾特征。

In [ ]:
%%stata
// 直方图
foreach c in 600519 66 600009{
	line rr date if stkcd==`c'
	ac rr if stkcd==`c'
	ac rr2 if stkcd==`c'
	qnorm rr if stkcd==`c'
	hist rr if stkcd==`c', normal
}

用 statsby 对每只股票分别执行 su rr, de，并把均值、标准差、偏度、峰度四个统计量逐只股票存入 stock_return_facts.dta。这是把分组描述性统计批量落盘的高效写法。

In [ ]:
%%stata
// 计算每只股票的收益率的描述性统计
statsby mean=r(mean) sd=r(sd) skew=r(skewness) kurt=r(kurtosis), by(stkcd) saving(stock_return_facts.dta, replace): su rr, de

借助 frame（Stata 16+）新开一个数据框打开刚才保存的统计结果，避免打乱主数据；对四只统计量再做一次汇总描述，即可看到全体股票收益率分布的典型事实：均值近零、偏度近零、峰度显著大于 3。

In [ ]:
%%stata
// 打开stock_return_facts.dta
frame create descriptive
frame change descriptive
use stock_return_facts.dta
su mean-kurt, de
rm stock_return_facts.dta